# FracAtlas Phase 4 sweep (Kaggle)

Thin runner for `configs/sweep_phase4.yaml`: 4 imbalance variants x 3 seeds, one job queue per GPU.
All logic is in the repo; see `docs/phase-4-imbalance.md`.

**Settings:** Accelerator **GPU T4 x2**, Internet **On**, *Add Input* -> the FracAtlas dataset.
Run with **Save Version -> Save & Run All** so it finishes in the background (~2.5 h).

In [ ]:
import os
import subprocess

WORKDIR = "/kaggle/working/repo"
if not os.path.isdir(WORKDIR):
    subprocess.run(
        [
            "git",
            "clone",
            "-q",
            "https://github.com/Akinchan-Jain-7/fracatlas-fracture-detection.git",
            WORKDIR,
        ],
        check=True,
    )
os.chdir(WORKDIR)
os.environ["FRACDET_DATA_DIR"] = "/kaggle/input"
!git log --oneline -1
!pip install -q -e .
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader; nproc

In [ ]:
# Optional GPU smoke test (~3 min, separate output folder). Set SMOKE = True to run it.
SMOKE = False
if SMOKE:
    !python scripts/sweep.py configs/sweep_phase4.yaml --gpus 0,1 --out-root /kaggle/working/smoke --set train.num_workers=2 train.epochs=1 debug.n_train=128 debug.n_val=64 paths.output_dir=/kaggle/working/smoke
    !python scripts/compare_runs.py configs/sweep_phase4.yaml --out /kaggle/working/smoke/phase4_imbalance

In [ ]:
# The sweep: 12 runs, ~2.5 h on T4 x2. A failed run is recorded and the rest continue.
!python scripts/sweep.py configs/sweep_phase4.yaml --gpus 0,1 --set train.num_workers=2

In [ ]:
# Validation comparison + the pre-registered decision
!python scripts/compare_runs.py configs/sweep_phase4.yaml

from IPython.display import Image, display

display(Image("outputs/sweeps/phase4_imbalance/compare.png"))

In [ ]:
# Keep best.pt (the winner may need a test evaluation); drop last.pt to save space
!rm -f outputs/runs/*/last.pt && du -sh outputs